In [1]:
%load_ext autoreload
%autoreload 2
import moabb
from moabb.datasets import *
from moabb.evaluations import WithinSessionEvaluation
from moabb.paradigms import P300
from hoda.hoda import HODA,BTTDA
from sklearn.pipeline import make_pipeline
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from mne.decoding import Scaler
import matplotlib.pyplot as plt
import seaborn as sns
from moabb.analysis.plotting import paired_plot, meta_analysis_plot, summary_plot
from sklearn.base import BaseEstimator, ClassifierMixin
from toeplitzlda.classification import ToeplitzLDA
from sklearn.svm import SVC
from moabb.analysis.meta_analysis import (  # noqa: E501
    compute_dataset_statistics,
    find_significant_differences,
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


2023-10-04 15:28:06.962498: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Tensorflow not install, you could not use those pipelines
To use the get_shape_from_baseconcar, InputShapeSetterEEG, BraindecodeDatasetLoaderyou need to install `braindecode`.`pip install braindecode` or Please refer to `https://braindecode.org`.


In [2]:
from matplotlib import style
plt.style.use('default')

In [3]:
tmin = 0
tmax=0.8
fmin=0.5
fmax = 16
sfreq = 32

paradigm = P300(resample=sfreq, tmin=tmin, tmax=tmax, fmin=fmin, fmax=fmax)
datasets = [
    #bi2013a(),
    #bi2014a(),
    #bi2014b(),
    #bi2015a(),
    #bi2015b(),
    BNCI2014009(),
    #BNCI2014009(),
    #BNCI2015003(),
    #DemonsP300(),
    #EPFLP300(),
    #Huebner2017(),
    #Huebner2018(),
    #Lee2019_ERP(),
    #Sosulski2019()

]
evaluation = WithinSessionEvaluation(
    paradigm=paradigm,
    datasets=datasets,
    suffix="hoda",
    overwrite=True,
    random_state=42,
    n_jobs=-1,
    #data_size=dict(
    #    policy='per_class',
    #    value=[10]
    #),
    #n_perms=[2],
)

BNCI2014009 has been renamed to BNCI2014_009. BNCI2014009 will be removed in version 0.7.
The dataset class name 'BNCI2014009' must be an abbreviation of its code 'BNCI2014-009'. See moabb.datasets.base.is_abbrev for more information.


In [4]:
import tensorly as tl
tl.set_backend('numpy', local_threadsafe=False)

In [5]:
import numpy as np

class ToeplitzLDAWrapper(BaseEstimator, ClassifierMixin):

    def fit(self, X, y=None):
        self.classes_ = np.unique(y)
        n_epochs, n_channels, n_samples = X.shape
        self.tlda_ = ToeplitzLDA(n_channels=n_channels,
                                 data_is_channel_prime=False)
        X = X.reshape(n_epochs, -1)
        return self.tlda_.fit(X, y)

    def decision_function(self, X):
        n_epochs, n_channels, n_samples = X.shape
        X = X.reshape(n_epochs, -1)
        return self.tlda_.decision_function(X)

    def predict(self, X):
        n_epochs, n_channels, n_samples = X.shape
        X = X.reshape(n_epochs, -1)
        return self.tlda_.predict(X)

    def predict_proba(self, X):
        n_epochs, n_channels, n_samples = X.shape
        X = X.reshape(n_epochs, -1)
        return self.tlda_.predict_proba(X)


def reshape(X, y=None):
    return X.reshape((X.shape[0],-1))


In [6]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.svm import SVC
from sklearn.feature_selection import SelectPercentile, f_classif
from mne.decoding import Scaler
from hoda.hoda import HODA
import cupy

pipelines = dict()


pipelines['HODA3'] = make_pipeline(
    #Scaler(scalings='mean', with_mean=False),
    HODA(
        max_iter=32,
        rank=[3,3],
        tol=1e-6,
        init ='svd',
        shrinkage=('lw','lw'),
        toeplitz=None,
        obj='rt',
        solver='lanczos',        
        verbose=False,
        taper=False,
        lasso=False,
        keep_train_info=False
    ),
    FunctionTransformer(reshape),
    LinearDiscriminantAnalysis(),
)
pipelines['HODA'] = make_pipeline(
    #Scaler(scalings='mean', with_mean=False),
    HODA(
        max_iter=32,
        rank=None,
        tol=1e-6,
        init ='svd',
        shrinkage=('lw','lw'),
        toeplitz=None,
        obj='rt',
        solver='lanczos',        
        verbose=False,
        taper=False,
        lasso=True,
        keep_train_info=False
    ),
    FunctionTransformer(reshape),
    LinearDiscriminantAnalysis(),
)
"""
pipelines['BTTDA'] = make_pipeline(
    #Scaler(scalings='mean', with_mean=False),
    BTTDA(
        n_blocks=4,
        hoda_params=dict(
            max_iter=32,
            rank=None,
            tol=1e-6,
            init ='svd',
            shrinkage=('lw','lw'),
            toeplitz=None,
            obj='rt',
            solver='lanczos',        
            verbose=False,
            taper=False,
            lasso=True,
            keep_train_info=False
        ),
    ),
    LinearDiscriminantAnalysis(shrinkage='auto', solver='lsqr')
)

pipelines['tLDA'] = make_pipeline(
    #Scaler(scalings='mean', with_mean=False),
    ToeplitzLDAWrapper()
)
"""

"\npipelines['BTTDA'] = make_pipeline(\n    #Scaler(scalings='mean', with_mean=False),\n    BTTDA(\n        n_blocks=4,\n        hoda_params=dict(\n            max_iter=32,\n            rank=None,\n            tol=1e-6,\n            init ='svd',\n            shrinkage=('lw','lw'),\n            toeplitz=None,\n            obj='rt',\n            solver='lanczos',        \n            verbose=False,\n            taper=False,\n            lasso=True,\n            keep_train_info=False\n        ),\n    ),\n    LinearDiscriminantAnalysis(shrinkage='auto', solver='lsqr')\n)\n\npipelines['tLDA'] = make_pipeline(\n    #Scaler(scalings='mean', with_mean=False),\n    ToeplitzLDAWrapper()\n)\n"

In [9]:
results = evaluation.process(pipelines)



BNCI2014-009-WithinSession:   0%|                                                                                                                                                                                      | 0/10 [00:00<?, ?it/s]

No hdf5_path provided, models will not be saved.
No hdf5_path provided, models will not be saved.
No hdf5_path provided, models will not be saved.
No hdf5_path provided, models will not be saved.
No hdf5_path provided, models will not be saved.
No hdf5_path provided, models will not be saved.




BNCI2014-009-WithinSession:  10%|█████████████████▍                                                                                                                                                            | 1/10 [00:22<03:24, 22.70s/it]

No hdf5_path provided, models will not be saved.
No hdf5_path provided, models will not be saved.
No hdf5_path provided, models will not be saved.
No hdf5_path provided, models will not be saved.
No hdf5_path provided, models will not be saved.
No hdf5_path provided, models will not be saved.




BNCI2014-009-WithinSession:  20%|██████████████████████████████████▊                                                                                                                                           | 2/10 [00:49<03:21, 25.25s/it]

No hdf5_path provided, models will not be saved.
No hdf5_path provided, models will not be saved.


ValueError: alpha=0.05 to strict, no components retained

In [ ]:
results

In [ ]:
stats = compute_dataset_statistics(results)
P, T = find_significant_differences(stats)
_ = summary_plot(P, T)

In [ ]:
_ = paired_plot(results, "HODA2", "HODA3")
